# Project: Linear regression three ways 📐
Fit the same model with (1) the **normal equations**, (2) **gradient descent**, and (3) scikit-learn, and check they agree. Then add **ridge** regularisation and trace how the coefficients shrink as λ grows (the regularisation path) on the diabetes dataset.

Topic: [[Linear Regression]], [[Overfitting and Regularization]] · guide note: [[Project - Linear Regression Three Ways]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import train_test_split
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
data = load_diabetes(); X, y = data.data, data.target
X = (X - X.mean(0)) / X.std(0); y = y - y.mean()                # standardise (no intercept needed after centring)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
print(X.shape, data.feature_names)

## 1. Normal equations (with ridge)\n$\hat w = (X^\top X + \lambda I)^{-1}X^\top y$. Use `np.linalg.solve`. With $\lambda = 0$ it's ordinary least squares.

In [ ]:
def fit_normal(X, y, lam=0.0):
    # TODO 1: closed-form (ridge) solution
    raise NotImplementedError  # TODO 1

## 2. Gradient descent\nMinimise $\frac1n\lVert Xw-y\rVert^2 + \frac{\lambda}{n}\lVert w\rVert^2$ (same minimiser as the normal equations, scaled by $1/n$): gradient $\frac2n X^\top(Xw-y) + \frac{2\lambda}{n}w$. Start from zeros.

In [ ]:
def fit_gd(X, y, lam=0.0, lr=0.1, steps=5000):
    # TODO 2: plain gradient descent on the (ridge) squared loss
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_lr():
    w_ne, w_gd = fit_normal(X_tr, y_tr), fit_gd(X_tr, y_tr)
    w_sk = LinearRegression(fit_intercept=False).fit(X_tr, y_tr).coef_
    r_ne, r_sk = fit_normal(X_tr, y_tr, 10.0), Ridge(alpha=10.0, fit_intercept=False).fit(X_tr, y_tr).coef_
    return np.allclose(w_ne, w_sk, atol=1e-6) and np.allclose(w_gd, w_sk, atol=0.5) and np.allclose(r_ne, r_sk, atol=1e-6) and np.allclose(fit_gd(X_tr, y_tr, 10.0), r_sk, atol=0.5)
check("three ways agree", _t_lr, "Normal equations must match sklearn exactly; gradient descent to within 0.5 (it converges slowly on correlated features).")
if ready("three ways agree"):
    mse = lambda w: np.mean((X_te @ w - y_te) ** 2)
    lams = np.logspace(-2, 4, 40); path = np.array([fit_normal(X_tr, y_tr, l) for l in lams]); test = [mse(w) for w in path]
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    for j, name in enumerate(data.feature_names): ax[0].semilogx(lams, path[:, j], label=name)
    ax[0].set_xlabel("λ"); ax[0].set_title("ridge path: coefficients shrink to 0"); ax[0].legend(fontsize=7, ncol=2)
    ax[1].semilogx(lams, test); ax[1].set_xlabel("λ"); ax[1].set_title("test MSE"); plt.show()
    best = lams[int(np.argmin(test))]; print(f"OLS test MSE {mse(path[0]):.0f}; best λ = {best:.1f} with test MSE {min(test):.0f}")
    print("biggest |coefficients| at λ=0:", sorted(zip(np.abs(path[0]).round(1), data.feature_names), reverse=True)[:3])

<details><summary>▶ Solution</summary>

```python
def fit_normal(X, y, lam=0.0):
    return np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)
```

```python
def fit_gd(X, y, lam=0.0, lr=0.1, steps=5000):
    w = np.zeros(X.shape[1]); n = len(y)
    for _ in range(steps):
        grad = 2 / n * X.T @ (X @ w - y) + 2 * lam * w / n
        w -= lr * grad
    return w
```
</details>

## Stretch goals
- Why does GD converge slowly here? Compute the condition number of $X^\top X$ (s1 and s2 are strongly correlated) and relate it to [[Project - Optimizer Race]].
- Lasso path with `sklearn.linear_model.lasso_path`: coefficients hit exactly 0 one by one.
- Choose λ by 5-fold cross-validation instead of the test set ([[Cross-Validation and Model Selection]]).